In [1]:
%load_ext autoreload
%autoreload 2
%autosave 30
%matplotlib inline

Autosaving every 30 seconds


# Comparison and validation of FID implementation
As we used an autoencoder for the FID computation this notebook explores if this method can actually give insighs to the realness of the post-processed forecasts

For this, we will:
- Compute the FID for different post prosessing models
  - and explore relationships to the crps and other error measures
- Use different specification of the autoencoder
  - to explore how the size of the hidden dimension influences results
- Test if the FID increases when non-realistic forecasts are presented
  - for this we can use the averaged forecasts of the ensemble which should be "too smooth"
  - inject some sort of noise to the forecasts

In [2]:
import re
from itertools import product

import hvplot.polars  # noqa: F401
import lightning as L
import polars as pl
import torch
import xarray as xr
from einops import rearrange
from torch.utils.data import DataLoader, TensorDataset

from genpp import BASE_DIR
from genpp.data import (
    FC_VARS,
    FORECAST_ENS_PATH,
    TEST_PREDICTIONS,
    TRAIN_PREDICTIONS,
    VAL_PREDICTIONS,
)
from genpp.eval import best_encoders, best_models
from genpp.eval.FID import fid
from genpp.eval.utils import load_predictions_dataarray

## Load the encoder model

In [3]:
encoder = best_encoders.classifierEncoder.model

Ignored args: (), kwargs: {'use_rescaler': False, 'rescaler': None}
Loading channel_means from checkpoint
Loading channel_stds from checkpoint


## Generate the embeddings for the forecasts

In [4]:
predictions = []
_re = re.compile(r"predictions(?:_(?P<variant>.+))?$")

for model_name, models in best_models:
    for model in models:
        for file in list(model.model_dir.rglob("val_predictions*.zarr")):
            model_dict = {}
            model_dict["name"] = model_name
            model_dict["tag"] = model.tag
            key = model_name
            if model.tag:
                key += f"_{model.tag}"
            m = _re.search(file.stem)
            suffix = m.group("variant") if m and m.group("variant") else None
            if suffix:
                key += f"_{suffix}"
            model_dict["dependency_model"] = suffix
            model_dict["full_name"] = key
            model_dict["file"] = file

            print(f"Found prediction files for {key}")
            predictions.append(model_dict)

Found prediction files for emos_gca
Found prediction files for emos_ecc
Found prediction files for drn_gca
Found prediction files for drn_ecc
Found prediction files for chen_standard
Found prediction files for chen_chen_spatial_3
Found prediction files for chen_chen_spatial_5
Found prediction files for chen_chen_spatial_7
Found prediction files for fm_unet_std
Found prediction files for fm_uvit_std
Found prediction files for fm_uvit_abs
Found prediction files for engression_standard


In [5]:
predictions

[{'name': 'emos',
  'tag': None,
  'dependency_model': 'gca',
  'full_name': 'emos_gca',
  'file': PosixPath('/home/feik/GenPP/outputs/EMOS/2025-10-20_16-13-04/val_predictions_gca.zarr')},
 {'name': 'emos',
  'tag': None,
  'dependency_model': 'ecc',
  'full_name': 'emos_ecc',
  'file': PosixPath('/home/feik/GenPP/outputs/EMOS/2025-10-20_16-13-04/val_predictions_ecc.zarr')},
 {'name': 'drn',
  'tag': None,
  'dependency_model': 'gca',
  'full_name': 'drn_gca',
  'file': PosixPath('/home/feik/GenPP/outputs/DRN/2025-10-20_18-47-58/val_predictions_gca.zarr')},
 {'name': 'drn',
  'tag': None,
  'dependency_model': 'ecc',
  'full_name': 'drn_ecc',
  'file': PosixPath('/home/feik/GenPP/outputs/DRN/2025-10-20_18-47-58/val_predictions_ecc.zarr')},
 {'name': 'chen',
  'tag': 'standard',
  'dependency_model': None,
  'full_name': 'chen_standard',
  'file': PosixPath('/home/feik/GenPP/outputs/CHEN/2025-12-02_10-40-49/val_predictions.zarr')},
 {'name': 'chen',
  'tag': 'chen_spatial_3',
  'depende

In [6]:
trainer = L.Trainer(accelerator="auto", logger=False)

for model_dict in predictions:
    val_preds = load_predictions_dataarray(model_dict["file"])
    # Convert to latents
    val_preds_tensor = torch.tensor(val_preds.to_numpy()).float()
    val_preds_tensor = rearrange(
        val_preds_tensor,
        "prediction sample channel height width -> (prediction sample) channel height width",
    )
    dataset = TensorDataset(val_preds_tensor)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=False)
    latents: list[torch.Tensor] = trainer.predict(encoder, dataloader)  # type: ignore
    latents_cat = torch.cat(latents, dim=0)
    model_dict["latents"] = latents_cat

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A5000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1

Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


...

Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

## Compute the latents for the ground truth dataset
This is the ifs ensemble predictions.

In [7]:
splits = [("train", TRAIN_PREDICTIONS), ("val", VAL_PREDICTIONS), ("test", TEST_PREDICTIONS)]
gt_latents = []
for split, split_prediction in splits:
    gt = (
        xr.open_zarr(FORECAST_ENS_PATH)[FC_VARS]
        .to_dataarray("feature")
        .stack(prediction=("time", "prediction_timedelta"))
        .sel(prediction=split_prediction)
        .transpose("prediction", "number", "feature", "longitude", "latitude")
    )
    gt_tensor = torch.tensor(gt.to_numpy()).float()
    gt_tensor = rearrange(
        gt_tensor,
        "prediction number feature longitude latitude -> (prediction number) feature longitude latitude",
    )
    # Assert no NaNs
    assert not torch.isnan(gt_tensor).any()
    dataset = TensorDataset(gt_tensor)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=False)
    latents: list[torch.Tensor] = trainer.predict(encoder, dataloader)  # type: ignore
    latents_cat = torch.cat(latents, dim=0)
    gt_latents.append({"split": split, "latents": latents_cat})

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

In [8]:
rows = []
for gt1, gt2 in product(gt_latents, repeat=2):
    if gt1["split"] > gt2["split"]:
        continue
    elif gt1["split"] == gt2["split"]:
        rows.append(
            {
                "gt_model": gt1["split"],
                "model": gt2["split"],
                "fid": 0.0,
            }
        )
    else:
        fid_value = fid(features1=gt1["latents"], features2=gt2["latents"])
        rows.append(
            {
                "gt_model": gt1["split"],
                "model": gt2["split"],
                "fid": fid_value,
            }
        )
        rows.append(
            {
                "gt_model": gt2["split"],
                "model": gt1["split"],
                "fid": fid_value,
            }
        )
for gt, model in product(gt_latents, predictions):
    fid_value = fid(features1=model["latents"], features2=gt["latents"])
    rows.append(
        {
            "gt_model": gt["split"],
            "model": model["full_name"],
            "fid": fid_value,
        }
    )

# Clip values to be non-negative as these come from numerical errors
fid_df = pl.DataFrame(rows).with_columns(pl.col("fid").clip(0, None))
fid_df

/home/feik/GenPP/src/genpp/eval/FID/__init__.py:126: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean: np.typing.NDArray[Any] = scipy.linalg.sqrtm(sigma1.mm(sigma2).numpy())  # pyright: ignore[reportAssignmentType]


gt_model,model,fid
str,str,f64
"""train""","""train""",0.0
"""train""","""val""",0.0
"""val""","""train""",0.0
"""val""","""val""",0.0
"""test""","""train""",0.0
…,…,…
"""test""","""chen_chen_spatial_7""",17.610458
"""test""","""fm_unet_std""",8.636045
"""test""","""fm_uvit_std""",12.294443


## Notes on the following Plot

In the following we will compare the fid scores of the postprocessing models.\
Note that for all postprocessing models, the latents stem from the **validation set**.\
So the bad score for the other sets are somewhat expected.\
However the "real" forecasts should never have a worse score than the post-processed ones. This checks out and is a good sign.


In [9]:
fid_df.hvplot.heatmap(  # pyright: ignore[reportAttributeAccessIssue]
    x="gt_model",
    y="model",
    C="fid",
    cmap="Reds",
    title="FID Scores Between Models",
    width=800,
    height=600,
    colorbar=True,
)

:HeatMap   [gt_model,model]   (fid)

## Lets add some intentionlly "bad" forecasts
For this we use the validation set of real forecasts and apply some transformations such as:
- gaussian noise
- gaussian blur
- black rectangles
- swirl
- salt and pepper noise

This list is taken from the FID paper: [GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium](https://arxiv.org/abs/1706.08500) (Heusel et al., 2018).

But first lets start with the mean forecast

In [10]:
# Load the data
from torch.utils.data import DataLoader, TensorDataset

from genpp.data import FC_VARS

ens_mean = xr.open_dataset(BASE_DIR / "data" / "weatherbench2" / "ens_flat_agg.zarr")
val_slice = slice("2021-01-01", "2021-12-31")

ens_mean = (
    ens_mean.sel(statistic="mean")[FC_VARS]
    .stack(prediction=["time", "prediction_timedelta"])
    .sel(prediction=VAL_PREDICTIONS)
    .to_dataarray("feature")
    .transpose("prediction", "feature", "longitude", "latitude")
)
mean_fc_tensor = torch.tensor(ens_mean.values).float()
mean_fc_dataset = TensorDataset(mean_fc_tensor)
mean_fc_dataloader = DataLoader(mean_fc_dataset, batch_size=128, shuffle=False)

trainer = L.Trainer(logger=False)
ens_mean_preds = trainer.predict(encoder, mean_fc_dataloader)
ens_mean_preds = torch.cat(ens_mean_preds, dim=0)  # type: ignore

predictions.append(
    {
        "name": "ens_mean",
        "tag": None,
        "dependency_model": None,
        "full_name": "ens_mean",
        "file": None,
        "latents": ens_mean_preds,
    }
)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

In [11]:
for gt in gt_latents:
    fid_value = fid(
        features1=gt["latents"],
        features2=ens_mean_preds,  # type: ignore
    )
    rows.append(
        {
            "gt_model": gt["split"],
            "model": "ens_mean",
            "fid": fid_value,
        }
    )
# Clip values to be non-negative as these come from numerical errors
fid_df = pl.DataFrame(rows).with_columns(pl.col("fid").clip(0, None))
fid_df

/home/feik/GenPP/src/genpp/eval/FID/__init__.py:126: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean: np.typing.NDArray[Any] = scipy.linalg.sqrtm(sigma1.mm(sigma2).numpy())  # pyright: ignore[reportAssignmentType]


gt_model,model,fid
str,str,f64
"""train""","""train""",0.0
"""train""","""val""",0.0
"""val""","""train""",0.0
"""val""","""val""",0.0
"""test""","""train""",0.0
…,…,…
"""test""","""fm_uvit_abs""",9.329064
"""test""","""engression_standard""",54.742283
"""train""","""ens_mean""",23.815117


In [12]:
fid_df.hvplot.heatmap(  # pyright: ignore[reportAttributeAccessIssue]
    x="gt_model",
    y="model",
    C="fid",
    cmap="Reds",
    title="FID Scores Between Models",
    width=800,
    height=600,
    colorbar=True,
)

:HeatMap   [gt_model,model]   (fid)

### Few notes on this

If the Autoencoder is good at encoding forecasts it is probably also good at encoding smoothed forecasts.\
Instead of using an autoencoder use some classifier that can differentiate between forecasts and too smooth forecasts (i.e. mean forecasts)


### Now add some intentional noise

In [13]:
from torchvision.transforms import v2

from genpp.data.zarr_dataset import TransformTensorDataset

In [14]:
ens_fc = (
    xr.open_dataset(BASE_DIR / "data" / "weatherbench2" / "ifs_ens.zarr")[FC_VARS]
    .stack(prediction=["time", "prediction_timedelta"])
    .sel(prediction=VAL_PREDICTIONS)
    .to_dataarray("feature")
    .transpose("prediction", "number", "feature", "longitude", "latitude")
)

In [15]:
gaussian_noise = v2.GaussianNoise(mean=0.0, sigma=1, clip=False)  # p=1 by default
gaussian_blur = v2.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0))  # p=1 by default
black_rectangle = v2.RandomErasing(
    p=1,
    scale=(0.02, 0.33),
    ratio=(0.3, 3.3),
    value=tuple(ens_fc.mean(["prediction", "number", "longitude", "latitude"]).values),  # type: ignore
)

defects = [gaussian_noise, gaussian_blur, black_rectangle]
for defect in defects:
    name = "ifs_ens+" + defect.__class__.__name__
    ens_fc_tensor = rearrange(
        torch.tensor(ens_fc.values).float(),
        "prediction number feature longitude latitude -> (prediction number) feature longitude latitude",
    )
    ens_fc_ds = TransformTensorDataset(ens_fc_tensor, transform=defect)
    dl = DataLoader(ens_fc_ds, batch_size=265, shuffle=False)
    ens_noised_preds = trainer.predict(encoder, dl)
    ens_noised_preds = torch.cat(ens_noised_preds, dim=0)  # type: ignore

    predictions.append(
        {
            "name": "ifs_ens",
            "tag": defect.__class__.__name__,
            "dependency_model": None,
            "full_name": name,
            "file": None,
            "latents": ens_noised_preds,
        }
    )

    for gt in gt_latents:
        fid_value = fid(
            features1=gt["latents"],
            features2=ens_noised_preds,  # type: ignore
        )
        rows.append(
            {
                "gt_model": gt["split"],
                "model": name,
                "fid": fid_value,
            }
        )

fid_df = pl.DataFrame(rows).with_columns(pl.col("fid").clip(0, None))
fid_df

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

/home/feik/GenPP/src/genpp/eval/FID/__init__.py:126: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean: np.typing.NDArray[Any] = scipy.linalg.sqrtm(sigma1.mm(sigma2).numpy())  # pyright: ignore[reportAssignmentType]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

gt_model,model,fid
str,str,f64
"""train""","""train""",0.0
"""train""","""val""",0.0
"""val""","""train""",0.0
"""val""","""val""",0.0
"""test""","""train""",0.0
…,…,…
"""val""","""ifs_ens+GaussianBlur""",11.458427
"""test""","""ifs_ens+GaussianBlur""",11.507413
"""train""","""ifs_ens+RandomErasing""",2.363459


In [16]:
fid_df.hvplot.heatmap(  # pyright: ignore[reportAttributeAccessIssue]
    x="gt_model",
    y="model",
    C="fid",
    cmap="Reds",
    title="FID Scores Between Models",
    width=800,
    height=600,
    colorbar=True,
)

:HeatMap   [gt_model,model]   (fid)

## Compute the scores in latent space

In [17]:
import numpy as np

from genpp.data import OBSERVATIONS_FLAT_PATH
from genpp.models.loss import EnergyScore, EnsembleCRPS

In [18]:
# get the embeddings for the reforecast dataset
y_val = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)
    .sel(
        time=VAL_PREDICTIONS.get_level_values("time")
        + VAL_PREDICTIONS.get_level_values("prediction_timedelta")
    )
    .to_dataarray("feature")
    .sel(feature=FC_VARS)
    .transpose("time", "feature", "longitude", "latitude")
    .rename({"time": "prediction_time"})
    .assign_coords(
        prediction=("prediction_time", VAL_PREDICTIONS),
    )
    .swap_dims({"prediction_time": "prediction"})
)

# pass through the encoder
y_tensor = torch.tensor(y_val.to_numpy()).float()
# Assert no NaNs
assert not torch.isnan(y_tensor).any()
dataset = TensorDataset(y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False)
latents: list[torch.Tensor] = trainer.predict(encoder, dataloader)  # type: ignore
y_latents = torch.cat(latents, dim=0)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

In [24]:
scores = []
scores_t = []
scores_per_leadtime = []
CRPS = EnsembleCRPS(n_axis=-2)
ES = EnergyScore(clamp=False)

prediction_timedeltas = VAL_PREDICTIONS.get_level_values("prediction_timedelta").to_numpy()
td = np.unique(prediction_timedeltas)
td_h = [td / np.timedelta64(1, "h") for td in td]

for model in predictions:
    # the CRPS expexts input dim [b, n, d] for x and [b, d] for y
    if model["full_name"] == "ens_mean":
        continue
    latent_reshaped = rearrange(model["latents"], "(b n) d -> b n d", n=50)
    crps = CRPS(latent_reshaped, y_latents)
    energy_score = ES(latent_reshaped, y_latents)
    mse = torch.mean((latent_reshaped - rearrange(y_latents, "b d -> b 1 d")) ** 2, dim=-1)
    scores.append(
        {
            "model": model["full_name"],
            "crps": crps.mean(),
            "energy_score": energy_score.mean(),
            "mse": mse.mean(),
        }
    )

    for delta, delta_h in zip(td, td_h):
        mask = prediction_timedeltas == delta
        crps_t = crps[mask].mean()
        energy_score_t = energy_score[mask].mean()
        mse_t = mse[mask].mean()
        scores_t.append(
            {
                "model": model["full_name"],
                "prediction_timedelta": delta_h,
                "crps": crps_t,
                "energy_score": energy_score_t,
                "mse": mse_t,
            }
        )

scores_df = pl.DataFrame(scores)
scores_t_df = pl.DataFrame(scores_t)

In [31]:
p1 = scores_df.hvplot.bar(  # type: ignore[reportArgumentTypeMismatch]
    x="model",
    y="crps",
    title="CRPS on Val Set",
    width=400,
    height=400,
    rot=45,
)

p2 = scores_df.hvplot.bar(  # type: ignore[reportArgumentTypeMismatch]
    x="model",
    y="energy_score",
    title="Energy Score on Val Set",
    width=400,
    height=400,
    rot=45,
)

p3 = scores_df.hvplot.bar(  # type: ignore[reportArgumentTypeMismatch]
    x="model",
    y="mse",
    title="MSE on Val Set",
    width=400,
    height=400,
    rot=45,
)

(p1 + p2 + p3).cols(3)

:Layout
   .Bars.I   :Bars   [model]   (crps)
   .Bars.II  :Bars   [model]   (energy_score)
   .Bars.III :Bars   [model]   (mse)

In [32]:
p4 = scores_t_df.hvplot.line(  # type: ignore[reportArgumentTypeMismatch]
    x="prediction_timedelta",
    y="crps",
    by="model",
    title="CRPS on Val Set by Prediction Timedelta",
    width=400,
    height=500,
    legend="bottom",
    legend_cols=2,
)

p5 = scores_t_df.hvplot.line(  # type: ignore[reportArgumentTypeMismatch]
    x="prediction_timedelta",
    y="energy_score",
    by="model",
    title="Energy Score on Val Set by Prediction Timedelta",
    width=400,
    height=500,
    legend="bottom",
    legend_cols=2,
)

p6 = scores_t_df.hvplot.line(  # type: ignore[reportArgumentTypeMismatch]
    x="prediction_timedelta",
    y="mse",
    by="model",
    title="MSE on Val Set by Prediction Timedelta",
    width=400,
    height=500,
    legend="bottom",
    legend_cols=2,
)

(p4 + p5 + p6).cols(3)

:Layout
   .NdOverlay.I   :NdOverlay   [model]
      :Curve   [prediction_timedelta]   (crps)
   .NdOverlay.II  :NdOverlay   [model]
      :Curve   [prediction_timedelta]   (energy_score)
   .NdOverlay.III :NdOverlay   [model]
      :Curve   [prediction_timedelta]   (mse)